# 02 - Feature Engineering

## Purpose
Build the deterministic model table from raw monthly GEE observations. The reduced workload keeps all raw observation columns and adds only compact hydrology-focused calendar, lag, rolling, and difference features.

## Inputs
- `data/metadata/sample_index.csv`
- `data/raw/monthly/<sample_id>.csv`

## Outputs
- `data/processed/processed_data.csv`

## Notes
This notebook does not create learned anomaly scores, future targets, or model predictions.


## 1. Configure Feature Engineering Paths
Defines the project root, config path, and execution switch for deterministic feature engineering.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "config" / "project_config.yaml"
RUN_FEATURE_ENGINEERING = True

## 2. Build Processed Dataset
Reads all raw monthly CSVs, validates monthly continuity, and writes the reduced deterministic feature table.


In [2]:
from src.monthly_feature_engineering import build_processed_dataset

if RUN_FEATURE_ENGINEERING:
    processed = build_processed_dataset(CONFIG_PATH)
    print(processed.shape)
    display(processed.head())
else:
    print("RUN_FEATURE_ENGINEERING is False.")

(3234, 177)


,sample_id,grid_row,grid_col,latitude,longitude,month,ndvi_mean,evi_mean,ndmi_mean,ndwi_mean,...,water_probability_mean_difference_3m,flooded_vegetation_probability_mean_lag_1,flooded_vegetation_probability_mean_lag_3,flooded_vegetation_probability_mean_lag_6,flooded_vegetation_probability_mean_lag_12,flooded_vegetation_probability_mean_rolling_mean_3,flooded_vegetation_probability_mean_rolling_mean_6,flooded_vegetation_probability_mean_rolling_mean_12,flooded_vegetation_probability_mean_difference_1m,flooded_vegetation_probability_mean_difference_3m
0,sagil_1,0,0,2.341985,102.629998,2021-01-01,0.559763,0.445020,0.106943,-0.527780,...,NaN,NaN,NaN,NaN,NaN,0.034129,0.034129,0.034129,NaN,NaN
1,sagil_1,0,0,2.341985,102.629998,2021-02-01,0.455169,0.350538,0.100939,-0.444946,...,NaN,0.034129,NaN,NaN,NaN,0.034129,0.034129,0.034129,NaN,NaN
2,sagil_1,0,0,2.341985,102.629998,2021-03-01,0.419253,0.367311,0.051128,-0.422619,...,NaN,NaN,NaN,NaN,NaN,0.033250,0.033250,0.033250,NaN,NaN
3,sagil_1,0,0,2.341985,102.629998,2021-04-01,0.669163,0.591764,0.192576,-0.590714,...,-0.00213,0.032371,0.034129,NaN,NaN,0.031581,0.032431,0.032431,-0.001579,-0.003337
4,sagil_1,0,0,2.341985,102.629998,2021-05-01,0.719184,0.647537,0.246818,-0.618140,...,NaN,0.030792,NaN,NaN,NaN,0.037135,0.036383,0.036383,0.017449,NaN


## 3. Validate Processed Columns
Checks the processed shape and confirms that learned risk/target columns are not created in this notebook.


In [3]:
import pandas as pd
from src.validation import load_project_config

config = load_project_config(CONFIG_PATH)
processed_path = PROJECT_ROOT / config["paths"]["processed_data_csv"]
if processed_path.exists():
    processed = pd.read_csv(processed_path)
    forbidden_prefixes = ("target_", "predicted_", "actual_")
    forbidden_names = {
        "anomaly_magnitude",
        "pca_anomaly_magnitude",
        "signed_risk_score",
        "risk_direction",
        "risk_severity",
    }
    leaks = [
        column for column in processed.columns
        if column.startswith(forbidden_prefixes) or column in forbidden_names
    ]
    assert not leaks, f"Learned or future columns found in processed_data.csv: {leaks}"
    print("Processed data validation passed")
    print(processed.shape)
else:
    print(f"Processed dataset not found yet: {config['paths']['processed_data_csv']}")

Processed data validation passed
(3234, 177)
